# kNN enrichment of marker-high cells: rank vs intensity (Danenberg 2022 IMC)

**Background (CyTOF / C2S project).** In CyTOF, the rank encoding (the C2S "sentence") gave higher kNN enrichment than intensity.
kNN enrichment asks: for cells that are high in a marker, what fraction of their 30 profile neighbours are also high,
relative to the overall share of high cells? The definition is copied from the CyTOF code (`bc_knn_enrichment.py`):
- "high" = robust z > 1;
- neighbours = the 30 nearest in the encoded profile (Euclidean);
- score = mean over high cells of (fraction of neighbours that are high) / (share of high cells in the sample).

Encodings and data are the same as `ProfileVsPhysical.ipynb`:
- per-image robust z clipped to ±5, and the rank vector computed from it;
- tumour images with at least 200 cells;
- "high" is always defined from the intensity z, for both encodings.

**What is measured, per image and per marker M:**
- `E_prof`: enrichment among profile neighbours (M is part of the profile, as in the CyTOF code).
- `E_lto`: the same, with **M left out** of the profile used to find neighbours. This removes the part of the enrichment that is built in because M itself sets the distances.
- `E_type`: enrichment when each profile neighbour is replaced by a random cell of the same phenotype. This is what cell type alone gives.
- `E_phys`: enrichment among the 30 **physically** nearest cells, i.e. how clustered M-high cells are in the tissue. It does not depend on any encoding.

**Profile spaces:** `cytof6` (the 6 CyTOF-shared markers), `full36`, and the same random 6/12/18/24/31-marker sets as `ProfileVsPhysical` version 3 (same seed). Leave-one-out is run for `cytof6` and `full36` only.

## Reading (fixed before the run)
1. **Replication:** does rank beat intensity on `E_prof`? This uses the per-patient (rank − intensity) difference, averaged over markers, with a patient-bootstrap CI.
2. **Built-in part:** does the rank advantage survive leave-one-out (`E_lto`)? If it shrinks to ~0, the advantage mostly comes from M being in the space.
3. **Beyond cell type:** is `E_prof − E_type` also higher for rank? If not, rank's tighter grouping is cell-type grouping.
4. **Biology vs encoding:** across (image, marker), does `E_prof` track `E_phys` (Spearman) better for rank than for intensity (patient-bootstrap CI of the difference)?
   - If yes, rank's tighter grouping follows real tissue clustering of marker-high cells.
   - If no, it is an encoding effect.

In [ ]:
import os, glob, json, time
import numpy as np, pandas as pd
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr
from joblib import Parallel, delayed
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 20260927
OUT = "/kaggle/working"
COFACTOR = 1.0
K = 30
HIGH_Z = 1.0
MIN_HIGH = 10                  # a marker needs >= 10 high cells in an image to be scored there
MIN_CELLS, MAX_CELLS = 200, 6000
N_TYPE_NULL = 3
N_BOOT = 1000
SUBSET_SIZES = {6: 20, 12: 8, 18: 8, 24: 8, 31: 8}    # identical to ProfileVsPhysical v3
CYTOF6 = ["pan_cytokeratin", "cytokeratin_5", "cytokeratin_8_18", "sma",
          "estrogen_receptor_alpha", "ki_67"]
N_JOBS = os.cpu_count()

def find_input(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"missing input {name}"
    return hits[0]

## Functions

In [ ]:
def rank_vec(Zs):
    """0 = highest; ties by column order (stable sort), as in the CyTOF code."""
    order = np.argsort(-Zs, axis=1, kind="stable")
    R = np.empty_like(order)
    R[np.arange(len(Zs))[:, None], order] = np.arange(Zs.shape[1])
    return R.astype(np.float32)

def profile_knn(V, k, rng):
    D = cdist(V, V).astype(np.float32)
    D += rng.uniform(0, 1e-5, D.shape).astype(np.float32)     # random tie-breaking
    np.fill_diagonal(D, np.inf)
    return np.argpartition(D, k, axis=1)[:, :k]

def enrichment(P, h):
    """Mean over high cells of the fraction of their neighbours that are high, divided by the high share."""
    rate = h.mean()
    return h[P[h]].mean() / rate

def per_image(i, seed, xy_i, ph_i, Z_i, spaces, lto_spaces):
    rng = np.random.default_rng(seed)
    n = len(xy_i)
    if n > MAX_CELLS:
        keep = np.sort(rng.choice(n, MAX_CELLS, replace=False))
        xy_i, ph_i, Z_i = xy_i[keep], ph_i[keep], Z_i[keep]
        n = MAX_CELLS
    H = Z_i > HIGH_Z
    scored = np.flatnonzero((H.sum(0) >= MIN_HIGH) & (H.mean(0) < 0.9))
    if len(scored) == 0:
        return []
    by_ph = {p: np.flatnonzero(ph_i == p) for p in np.unique(ph_i)}
    self_ = np.arange(n)[:, None]

    def null_type(P):
        Q = np.empty_like(P); phP = ph_i[P]
        for p, idx in by_ph.items():
            m = phP == p
            Q[m] = idx[rng.integers(0, len(idx), m.sum())]
        bad = Q == self_
        Q[bad] = P[bad]
        return Q

    _, S = cKDTree(xy_i).query(xy_i, k=K + 1)
    S = S[:, 1:]
    e_phys = {m: enrichment(S, H[:, m]) for m in scored}
    rows = []
    for sname, cols in spaces.items():
        in_space = [m for m in cols if m in set(scored)]
        if not in_space:
            continue
        Zs = Z_i[:, cols]
        for enc in ("intensity", "rank"):
            P = profile_knn(Zs if enc == "intensity" else rank_vec(Zs), K, rng)
            Qs = [null_type(P) for _ in range(N_TYPE_NULL)]
            for m in in_space:
                h = H[:, m]
                rows.append(dict(img=i, space=sname, enc=enc, marker=int(m), variant="standard",
                                 E_prof=enrichment(P, h), E_type=np.mean([enrichment(Q, h) for Q in Qs]),
                                 E_phys=e_phys[m], n_high=int(h.sum()), rate=float(h.mean())))
    for sname in lto_spaces:
        cols = spaces[sname]
        for m in [m for m in cols if m in set(scored)]:
            rest = [c for c in cols if c != m]
            Zs = Z_i[:, rest]
            h = H[:, m]
            for enc in ("intensity", "rank"):
                P = profile_knn(Zs if enc == "intensity" else rank_vec(Zs), K, rng)
                rows.append(dict(img=i, space=sname, enc=enc, marker=int(m), variant="lto",
                                 E_prof=enrichment(P, h), E_type=enrichment(null_type(P), h),
                                 E_phys=e_phys[m], n_high=int(h.sum()), rate=float(h.mean())))
    return rows

def boot_mean(v, n_boot=N_BOOT, seed=SEED):
    v = np.asarray(v, float); v = v[~np.isnan(v)]
    r = np.random.default_rng(seed)
    bs = v[r.integers(0, len(v), (n_boot, len(v)))].mean(1)
    return v.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)

## Sanity check: random profiles must give enrichment ≈ 1; profiles that carry the marker must give > 1

In [ ]:
r = np.random.default_rng(0)
n = 1500
Zr = r.normal(size=(n, 6)).astype(np.float32)
h = Zr[:, 0] > 1
e_in = enrichment(profile_knn(Zr, K, r), h)                  # marker in space -> enriched
e_out = enrichment(profile_knn(Zr[:, 1:], K, r), h)          # marker left out, others independent -> ~1
print(f"marker in space: {e_in:.2f}   left out (independent markers): {e_out:.2f}")
assert e_in > 2 and abs(e_out - 1) < 0.15, "sanity failed"
print("sanity checks passed")

## Load data, cohort, encodings (same filters as ProfileVsPhysical)

In [ ]:
report = json.load(open(find_input("prep_report.json")))
cells = pd.read_parquet(find_input("cells.parquet"))
chan = list(report["channel_map"].values())
PROTEIN = [c for c in chan if c not in ("histone_h3", "dna1", "dna2")]
assert len(PROTEIN) == 36 and set(CYTOF6) <= set(PROTEIN)
cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
n_per = cells.groupby("image_name").size()
cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
print(f"{len(cells):,} cells, {cells.image_name.nunique()} images, {cells.metabric_id.nunique()} patients")

ph_code, ph_names = pd.factorize(cells.cell_phenotype, sort=True)
img_code, img_names = pd.factorize(cells.image_name, sort=True)
img_patient = cells.groupby("image_name").metabric_id.first().reindex(img_names).values
starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
xy = cells[["x", "y"]].values.astype(np.float64)
A = np.arcsinh(cells[PROTEIN].values.astype(np.float64) / COFACTOR)
Z = np.empty_like(A, dtype=np.float32)
for a, b in zip(starts[:-1], starts[1:]):
    med = np.median(A[a:b], 0)
    mad = np.median(np.abs(A[a:b] - med), 0)
    Z[a:b] = np.clip((A[a:b] - med) / (1.4826 * mad + 1e-6), -5, 5)
del A

rng = np.random.default_rng(SEED)                  # same draws as ProfileVsPhysical v3
col = {c: i for i, c in enumerate(PROTEIN)}
SPACES = {"cytof6": [col[c] for c in CYTOF6], "full36": list(range(36))}
seen = {tuple(sorted(SPACES["cytof6"]))}
for size, n_sub in SUBSET_SIZES.items():
    got = 0
    while got < n_sub:
        s = tuple(sorted(rng.choice(36, size, replace=False).tolist()))
        if s not in seen:
            seen.add(s)
            SPACES[f"rand{size}_{got:02d}"] = list(s)
            got += 1
def size_of(space):
    return 36 if space == "full36" else 6 if space == "cytof6" else int(space[4:].split("_")[0])
json.dump({s: [PROTEIN[i] for i in v] for s, v in SPACES.items()}, open(f"{OUT}/spaces.json", "w"), indent=1)
LTO = ("cytof6", "full36")

## Run all images

In [ ]:
t0 = time.time()
res = Parallel(n_jobs=N_JOBS, verbose=5)(
    delayed(per_image)(i, SEED + i, xy[a:b], ph_code[a:b], Z[a:b], SPACES, LTO)
    for i, (a, b) in enumerate(zip(starts[:-1], starts[1:])))
print(f"per-image: {(time.time() - t0) / 60:.1f} min")
d = pd.DataFrame([r for rows in res for r in rows])
d["patient"] = img_patient[d.img.values]
d["marker_name"] = np.array(PROTEIN)[d.marker.values]
d["size"] = d.space.map(size_of)
d.to_parquet(f"{OUT}/enrich_per_image.parquet", index=False)
print(d.shape)

## Summaries

In [ ]:
d["beyond_type"] = d.E_prof - d.E_type
def contrast(sub, value):
    """Per patient: mean over (image, marker, space) of rank − intensity; then patient bootstrap."""
    w = sub.pivot_table(index=["patient", "img", "space", "marker"], columns="enc", values=value)
    w = (w["rank"] - w["intensity"]).groupby("patient").mean()
    return boot_mean(w.values)

rows = []
for space in LTO:
    for variant in ("standard", "lto"):
        sub = d[(d.space == space) & (d.variant == variant)]
        for value in ("E_prof", "beyond_type"):
            row = dict(space=space, variant=variant, value=value)
            for enc in ("intensity", "rank"):
                pp = sub[sub.enc == enc].groupby("patient")[value].mean()
                row[enc], row[f"{enc}_lo"], row[f"{enc}_hi"] = boot_mean(pp.values)
            row["diff"], row["diff_lo"], row["diff_hi"] = contrast(sub, value)
            rows.append(row)
        pp = sub[sub.enc == "intensity"].groupby("patient").E_phys.mean()
        rows.append(dict(space=space, variant=variant, value="E_phys", intensity=boot_mean(pp.values)[0]))
summary = pd.DataFrame(rows)
summary.to_csv(f"{OUT}/enrich_summary.csv", index=False)
print(summary.round(3).to_string(index=False))

# per marker (full36 and cytof6, both variants)
pm = (d[d.space.isin(LTO)].groupby(["space", "variant", "marker_name", "enc"])[["E_prof", "E_type", "E_phys"]]
      .mean().unstack("enc"))
pm.columns = [f"{a}_{b}" for a, b in pm.columns]
pm = pm.reset_index()
pm["rank_minus_int"] = pm.E_prof_rank - pm.E_prof_intensity
pm.to_csv(f"{OUT}/enrich_per_marker.csv", index=False)
print(pm[(pm.space == "full36") & (pm.variant == "standard")].sort_values("rank_minus_int")
      [["marker_name", "E_prof_intensity", "E_prof_rank", "rank_minus_int", "E_phys_intensity"]].round(2).to_string(index=False))

# panel-size curve (standard)
curve = []
cur = d[(d.variant == "standard") & (d.space.str.startswith("rand") | (d.space == "full36"))]
for size, g in cur.groupby("size"):
    for value in ("E_prof", "beyond_type"):
        row = dict(size=size, value=value)
        for enc in ("intensity", "rank"):
            row[enc] = boot_mean(g[g.enc == enc].groupby("patient")[value].mean().values)[0]
        row["diff"], row["diff_lo"], row["diff_hi"] = contrast(g, value)
        curve.append(row)
curve = pd.DataFrame(curve)
curve.to_csv(f"{OUT}/enrich_panel_curve.csv", index=False)
print(curve.round(3).to_string(index=False))

# does profile enrichment track physical clustering? Spearman over (image, marker); patient bootstrap
corr = []
for space in LTO:
    for variant in ("standard", "lto"):
        sub = d[(d.space == space) & (d.variant == variant)]
        w = sub.pivot_table(index=["patient", "img", "marker"], columns="enc", values=["E_prof", "E_phys"])
        w = w.dropna()
        pats = w.index.get_level_values("patient").values
        up = np.unique(pats)
        idx = {p: np.flatnonzero(pats == p) for p in up}
        def rho(ix):
            x = w.iloc[ix]
            return (spearmanr(x[("E_prof", "rank")], x[("E_phys", "rank")])[0],
                    spearmanr(x[("E_prof", "intensity")], x[("E_phys", "intensity")])[0])
        full = rho(np.arange(len(w)))
        rb = np.random.default_rng(SEED)
        bs = []
        for _ in range(300):
            ix = np.concatenate([idx[p] for p in rb.choice(up, len(up))])
            a, b = rho(ix); bs.append(a - b)
        corr.append(dict(space=space, variant=variant, rho_rank=full[0], rho_intensity=full[1],
                         diff=full[0] - full[1], diff_lo=np.percentile(bs, 2.5), diff_hi=np.percentile(bs, 97.5)))
corr = pd.DataFrame(corr)
corr.to_csv(f"{OUT}/enrich_phys_corr.csv", index=False)
print(corr.round(3).to_string(index=False))

## Answers to the pre-set questions, and figures

In [ ]:
def get(space, variant, value):
    return summary[(summary.space == space) & (summary.variant == variant) & (summary.value == value)].iloc[0]
answers = {}
for space in LTO:
    s_std, s_lto = get(space, "standard", "E_prof"), get(space, "lto", "E_prof")
    s_bt = get(space, "standard", "beyond_type")
    c = corr[(corr.space == space) & (corr.variant == "standard")].iloc[0]
    answers[space] = {
        "1_rank_beats_intensity": bool(s_std.diff_lo > 0),
        "1_diff": [s_std["diff"], s_std.diff_lo, s_std.diff_hi],
        "2_survives_leave_one_out": bool(s_lto.diff_lo > 0),
        "2_diff_lto": [s_lto["diff"], s_lto.diff_lo, s_lto.diff_hi],
        "3_rank_higher_beyond_type": bool(s_bt.diff_lo > 0),
        "3_diff_beyond_type": [s_bt["diff"], s_bt.diff_lo, s_bt.diff_hi],
        "4_rank_tracks_tissue_better": bool(c.diff_lo > 0),
        "4_rho_rank_vs_intensity": [c.rho_rank, c.rho_intensity, c.diff_lo, c.diff_hi],
    }
json.dump(answers, open(f"{OUT}/summary_enrich.json", "w"), indent=1, default=float)
print(json.dumps(answers, indent=1, default=float))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, variant in zip(axes, ("standard", "lto")):
    g = pm[(pm.space == "full36") & (pm.variant == variant)].sort_values("E_prof_intensity")
    y = np.arange(len(g))
    ax.scatter(g.E_prof_intensity, y, label="intensity", color="C0")
    ax.scatter(g.E_prof_rank, y, label="rank", color="C1")
    ax.scatter(g.E_phys_intensity, y, label="in tissue (physical)", color="k", marker="|", s=80)
    ax.set_yticks(y); ax.set_yticklabels(g.marker_name, fontsize=7)
    ax.axvline(1, color="grey", lw=0.8)
    ax.set_xlabel("kNN enrichment of marker-high cells")
    ax.set_title(f"full36, {'marker in profile' if variant == 'standard' else 'marker left out'}")
axes[0].legend()
fig.tight_layout(); fig.savefig(f"{OUT}/fig_enrich_markers.png", dpi=150)

fig, ax = plt.subplots(figsize=(6, 4.5))
g = curve[curve.value == "E_prof"].sort_values("size")
ax.plot(g["size"], g["diff"], "o-", color="k")
ax.fill_between(g["size"], g.diff_lo, g.diff_hi, color="k", alpha=0.2)
ax.axhline(0, color="grey", lw=0.8); ax.axvline(31, color="grey", ls=":", lw=0.8)
ax.set_xlabel("markers in panel (dotted = CyTOF panel size)")
ax.set_ylabel("rank − intensity (kNN enrichment)")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_enrich_curve.png", dpi=150)
print("done")